# Module 3 — Camera Geometry and Robotic Perception
## Unit 8 — Mini Project: See the Fruit, Place It in the World
### Lesson 8.2 — Building the Perception → World Pipeline

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*

In [ ]:
import numpy as np
K=np.array([[800,0,320],[0,800,240],[0,0,1.0]])
def distort(xn,yn,k1=0,k2=0,k3=0,p1=0,p2=0):
    r2=xn*xn+yn*yn; rad=1+k1*r2+k2*r2**2+k3*r2**3
    return xn*rad+2*p1*xn*yn+p2*(r2+2*xn*xn), yn*rad+p1*(r2+2*yn*yn)+2*p2*xn*yn
def undistort(u,v,K,dist,iters=12):
    xd=(u-K[0,2])/K[0,0]; yd=(v-K[1,2])/K[1,1]
    k1,k2,p1,p2,k3=dist
    xn=np.array(xd,float); yn=np.array(yd,float)
    for _ in range(iters):
        dx,dy=distort(xn,yn,k1,k2,k3,p1,p2); xn=xn+(xd-dx); yn=yn+(yd-dy)
    return float(xn),float(yn)
def deproject(xn,yn,Z): return np.array([xn*Z,yn*Z,Z])
def se3(R,t):
    T=np.eye(4); T[:3,:3]=R; T[:3,3]=t; return T
def transform(T,P): return (T@np.append(np.asarray(P,float),1.0))[:3]
def see_fruit_place_in_world(pixel,Z,K,dist,T_ac,T_wa):
    xn,yn=undistort(pixel[0],pixel[1],K,dist)
    Pc=deproject(xn,yn,Z); T_wc=T_wa@T_ac
    return transform(T_wc,Pc), Pc, T_wc
T_ac=se3(np.eye(3),[0,0,0.1]); T_wa=se3(np.eye(3),[1.0,0.5,0.0])
Pw,Pc,T_wc=see_fruit_place_in_world((480,160),0.3,K,np.zeros(5),T_ac,T_wa)
print('P_c:',np.round(Pc,3),' P_w:',np.round(Pw,3))

## Coding Exercise (§8)
Per-stage identity tests + canonical composition; distortion shifts only undistort stage.

In [ ]:
# YOUR CODE HERE
import numpy as np
assert np.allclose(Pw,[1.06,0.47,0.4])
# identity extrinsics => P_w == P_c
Pw_id,Pc_id,_=see_fruit_place_in_world((480,160),0.3,K,np.zeros(5),np.eye(4),np.eye(4))
assert np.allclose(Pw_id,Pc_id)
# distortion changes only the undistort output
xn0,_=undistort(480,160,K,np.zeros(5)); xn1,_=undistort(480,160,K,np.array([-0.2,0,0,0,0.0]))
assert not np.isclose(xn0,xn1)
print('stages verified; canonical P_w correct.')

## Check your work

In [ ]:
import numpy as np
assert np.allclose(Pw,[1.06,0.47,0.4])
assert np.allclose(Pc,[0.06,-0.03,0.3])
print('All checks passed.')